# 04 Generation

`src.generation` RAG 파이프라인을 구동하고 `src.evaluation` 답변 생성 품질을 hit / recall / bert score 기준 평가.



## Colab Setup

In [ ]:
!nvidia-smi

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/songhahyun/finance-terms-rag-chatbot.git'
REPO_BRANCH = 'dev'
REPO_DIR = Path('/content/finance-terms-rag-chatbot')

In [ ]:
!git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
print('cwd =', Path.cwd())

In [ ]:
# Python deps
!pip install -q -U pip
!pip install -q -r requirements.txt

# Ollama install + serve
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip -q install pandas tqdm

# Ollama pull
!ollama pull llama3.2:3b

os.environ['OLLAMA_BASE_URL'] = 'http://127.0.0.1:11434'
os.environ['OLLAMA_MODEL'] = 'llama3.2:3b'

print('Ollama ready:', os.environ['OLLAMA_BASE_URL'])

In [ ]:
from google.colab import userdata

REQUIRED_SECRETS = [
    "OPENAI_API_KEY",
    "NCP_APIGW_API_KEY",
    "CLOVASTUDIO_API_KEY",
    "HF_TOKEN",
    "WANDB_API_KEY",
]

missing = []

for name in REQUIRED_SECRETS:
    value = userdata.get(name)
    if value:
        os.environ[name] = value
    else:
        missing.append(name)

if missing:
    raise RuntimeError(
        "Colab Secrets에 다음 값을 추가하고 Notebook access를 켜세요: "
        + ", ".join(missing)
    )

## Run answer generation

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

print(f"Project root: {ROOT}")

In [ ]:
import pandas as pd

from src.common.config import get_settings
from src.evaluation.generation_pipeline import run_generation_experiment

settings = get_settings()

# EVAL_CSV_PATH = ROOT / "data" / "eval" / "testset" / "golden_testset_v2.csv"
EVAL_CSV_PATH = ROOT / "data" / "eval" / "testset" / "test.csv"
CHUNK_JSON_PATH = settings.default_chunk_json_path
BM25_INDEX_PATH = CHUNK_JSON_PATH.with_name("bm25_index.pkl")
OUTPUT_DIR = ROOT / "data" / "eval" / "outputs" / "generation_v2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

USE_WEAVE = True
WEAVE_PROJECT = "finance-terms-rag-generation"
WEAVE_LOG_CONTEXTS = True

EXPERIMENTS = [
    {
        "name": "hybrid_clova_bge-m3",
        "retrieval_mode": "hybrid",
        "dense_provider": "clova",
        "dense_model_name": "bge-m3",
        "dense_collection_name": "docs_clova",
        "dense_persist_directory": str(settings.chroma_clova_dir),
    },
    {
        "name": "dense_clova_bge-m3",
        "retrieval_mode": "dense",
        "dense_provider": "clova",
        "dense_model_name": "bge-m3",
        "dense_collection_name": "docs_clova",
        "dense_persist_directory": str(settings.chroma_clova_dir),
    },
    {
        "name": "hybrid_openai_text-embedding-3-small",
        "retrieval_mode": "hybrid",
        "dense_provider": "openai",
        "dense_model_name": "text-embedding-3-small",
        "dense_collection_name": "docs_openai",
        "dense_persist_directory": str(settings.chroma_openai_dir),
    },
]

print("Eval CSV:", EVAL_CSV_PATH)
print("Chunk JSON:", CHUNK_JSON_PATH)
print("BM25 index PKL:", BM25_INDEX_PATH)
print("Output Dir:", OUTPUT_DIR)
print("Generation model:", settings.ollama_model)
print("Use Weave:", USE_WEAVE)
print("Weave project:", WEAVE_PROJECT)


In [ ]:
all_results = {}
summary_rows = []

for exp in EXPERIMENTS:
    output_path = OUTPUT_DIR / f"{exp['name']}.csv"
    result_df = run_generation_experiment(
        experiment_name=exp["name"],
        eval_csv_path=EVAL_CSV_PATH,
        chunk_json_path=CHUNK_JSON_PATH,
        output_csv_path=output_path,
        retrieval_mode=exp["retrieval_mode"],
        dense_provider=exp["dense_provider"],
        dense_model_name=exp["dense_model_name"],
        dense_collection_name=exp["dense_collection_name"],
        dense_persist_directory=exp["dense_persist_directory"],
        bm25_index_path=BM25_INDEX_PATH,
        ollama_model=settings.ollama_model,
        ollama_base_url=settings.ollama_base_url,
        ollama_timeout=settings.ollama_timeout,
        k=5,
        language="ko",
        use_weave=USE_WEAVE,
        weave_project=WEAVE_PROJECT,
        weave_log_contexts=WEAVE_LOG_CONTEXTS,
    )
    all_results[exp["name"]] = output_path

    summary_rows.append(
        {
            "experiment": exp["name"],
            "rows": len(result_df),
            "avg_recall": float(result_df["recall"].mean()) if len(result_df) else 0.0,
            "avg_mrr": float(result_df["mrr"].mean()) if len(result_df) else 0.0,
            "avg_stage_1_retrieval_bm25_latency_sec": float(result_df["stage_1_retrieval_bm25_latency_sec"].mean()) if len(result_df) else 0.0,
            "avg_stage1_1_retrieval_dense_latency_sec": float(result_df["stage1_1_retrieval_dense_latency_sec"].mean()) if len(result_df) else 0.0,
            "avg_stage_1_retrieval_fusion_latency_sec": float(result_df["stage_1_retrieval_fusion_latency_sec"].mean()) if len(result_df) else 0.0,
            "avg_stage_1_retrieval_total_latency_sec": float(result_df["stage_1_retrieval_total_latency_sec"].mean()) if len(result_df) else 0.0,
            "avg_stage_2_generation_latency_sec": float(result_df["stage_2_generation_latency_sec"].mean()) if len(result_df) else 0.0,
            "avg_total_latency_sec": float(result_df["total_latency_sec"].mean()) if len(result_df) else 0.0,
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_output_path = OUTPUT_DIR / "generation_experiment_summary.csv"
summary_df.to_csv(summary_output_path, index=False, encoding="utf-8-sig")

print("Saved output files:")
for name, path in all_results.items():
    print(f"- {name}: {path}")
print(f"- summary: {summary_output_path}")

summary_df
